# 15 — Observability, Structured Logging, Metrics, and SLOs

Production systems need sufficient evidence to diagnose failures and measure reliability. Learn structured logs, tracing identifiers, latency summaries, and service-level objectives.

*Advanced · 90–150 min*

## You will learn
- Design concise contextual logging without leaking secrets
- Record counts, errors, and latency histograms
- Differentiate SLIs, SLOs, and alerts
- Understand trace and correlation identifiers

## Examples

### 1. Structured log records

Log events as machine-parseable fields rather than mixing operational metrics into unstructured paragraphs. The example outputs a JSON record to a string rather than writing external logs. Never log credentials or full sensitive payloads.

In [ ]:
import json
from datetime import datetime, timezone

def event_log(event, status, request_id):
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event": event,
        "status": status,
        "request_id": request_id,
    }
    return json.dumps(record, sort_keys=True)

record = event_log("import", "ok", "req-001")
print(json.loads(record)["status"])

**Review:** Why is correlation ID preferable to logging a user's entire request?

### 2. Latency measurements and monotonic clocks

Use time.perf_counter for elapsed durations, not wall-clock timestamps. Don't interpret a handful of toy measurements as production performance. Count attempts and failures separately from successful outcomes.

In [ ]:
import time
def measure(func, *args):
    started = time.perf_counter()
    try:
        value = func(*args)
        return {"ok": True, "value": value, "duration": time.perf_counter() - started}
    except Exception:
        return {"ok": False, "duration": time.perf_counter() - started}

print(measure(sum, [1, 2, 3]))

**Review:** How could you avoid hiding a useful exception while still recording a failure metric?

### 3. Aggregate latency percentiles cautiously

A percentile describes a portion of measured operations. Be careful with small samples and inconsistent measurement populations. The following exact nearest-rank percentile is educational, not a production metrics library.

In [ ]:
import math

def percentile_nearest_rank(values, p):
    if not values or not 0 < p <= 100:
        raise ValueError("nonempty values and percentile 0 < p <= 100 required")
    ordered = sorted(values)
    index = math.ceil(p / 100 * len(ordered)) - 1
    return ordered[index]

latencies_ms = [10, 12, 12, 14, 19, 35, 41, 80, 100, 130]
print("p90:", percentile_nearest_rank(latencies_ms, 90))

**Review:** Why is an average an incomplete latency objective?

### 4. SLIs, SLOs and error budgets

An SLI is an observable metric, such as successful requests divided by total requests. An SLO is a target for that indicator over a specific window. An error budget is the allowed failure portion of the request volume in that window.

In [ ]:
def availability_sli(successes, total):
    if total <= 0 or not 0 <= successes <= total:
        raise ValueError("invalid request counts")
    return successes / total

successes, total = 999, 1000
sli = availability_sli(successes, total)
target = 0.995
print(f"SLI={sli:.3%}, meets target={sli >= target}")

**Review:** What happens to an SLO calculation if traffic is highly uneven across days?

## Try it (optional)

Small open experiments to explore the topic. The **graded** exercises for this topic, with fill-in-the-blank code and automatic checks, are in the [workbook](../exercises/04_production_systems_exercises.ipynb).

**Exercise 1.** Write a JSON event logger that redacts specified field names.

In [ ]:
# Exercise 1
# TODO: implement and test.

**Exercise 2.** Create a latency summary with count, min, median, p95, and max.

In [ ]:
# Exercise 2
# TODO: implement and test.

**Exercise 3.** Build an availability SLI calculator and test invalid count inputs.

In [ ]:
# Exercise 3
# TODO: implement and test.

**Exercise 4.** Explain how an alert differs from a dashboard and when each is appropriate.

In [ ]:
# Exercise 4
# TODO: implement and test.

## Common mistakes

- Never put tokens or personal records in diagnostic logs.
- Latency and error metrics must define a measurement window.
- Avoid overclaiming conclusions from a tiny or biased sample.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** An SLO is…

   - **a)** A target for a metric (e.g. 99.9% success)
   - **b)** A measured metric
   - **c)** An alert

**2.** A correlation ID lets you…

   - **a)** Follow one request across logs and services
   - **b)** Encrypt logs
   - **c)** Count users

**3.** Good alerts fire on…

   - **a)** Every single error
   - **b)** CPU spikes only
   - **c)** Sustained user-visible problems

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/15', {1: '20feb57c6faf', 2: 'b86084f78462', 3: 'aa34ad9c656e'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [16 — API Contracts, Versioning, Pagination, and Evolution](16_api_design_versioning_contracts.ipynb)